# Spoken — AI-generated speech detection (run on Colab)

Run these cells **in order**, top to bottom (Shift+Enter on each). Each cell has a comment explaining what it does.

At the end you'll download a `spoken_results.zip` file — send that back in the chat and Claude will continue from there (generalization experiment + report).

In [ ]:
# 1) Clone the project repo and switch to the working branch
!git clone https://github.com/qusaiAboSondos/spoken.git
%cd spoken
!git checkout claude/project-step-by-step-jx5y1a

In [ ]:
# 2) Install python dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3) Download ASVspoof2019 LA (train+dev). This is ~7-8GB, takes a few minutes on Colab's fast connection.
!mkdir -p data/raw
!wget -q --show-progress https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip -O data/raw/LA.zip
!unzip -q data/raw/LA.zip -d data/raw/ASVspoof2019
!ls data/raw/ASVspoof2019/LA

In [ ]:
# 4) Copy the official protocol files into data/protocols/
!cp data/raw/ASVspoof2019/LA/ASVspoof2019_LA_cm_protocols/*.txt data/protocols/
!ls data/protocols/

In [ ]:
# 5) Subsample to a manageable size (1500/class for train, 500/class for dev)
!python -m src.subsample_protocol \
  --protocol data/protocols/ASVspoof2019.LA.cm.train.trn.txt \
  --n-per-class 1500 --output data/protocols/train_subset.txt

!python -m src.subsample_protocol \
  --protocol data/protocols/ASVspoof2019.LA.cm.dev.trl.txt \
  --n-per-class 500 --output data/protocols/dev_subset.txt

In [ ]:
# 6) Extract features (MFCC, LFCC, spectral). This is the slowest step — grab a coffee.
!python -m src.extract_features \
  --protocol data/protocols/train_subset.txt \
  --audio-dir data/raw/ASVspoof2019/LA/ASVspoof2019_LA_train/flac \
  --features mfcc lfcc spectral --output data/features/train.npz

!python -m src.extract_features \
  --protocol data/protocols/dev_subset.txt \
  --audio-dir data/raw/ASVspoof2019/LA/ASVspoof2019_LA_dev/flac \
  --features mfcc lfcc spectral --output data/features/dev.npz

In [ ]:
# 7) Run all experiments (MFCC+SVM, LFCC+SVM, MFCC+RF, feature combinations) and compare them
!python -m src.run_experiment \
  --train-features data/features/train.npz \
  --test-features data/features/dev.npz \
  --output-dir results/

import pandas as pd
pd.read_csv('results/comparison.csv')

In [ ]:
# 8) Package everything needed to continue (results, cached features, and the exact subsets used)
# and download it. Send the downloaded spoken_results.zip back in the chat.
!zip -rq spoken_results.zip results data/features data/protocols/train_subset.txt data/protocols/dev_subset.txt

from google.colab import files
files.download('spoken_results.zip')